# BERT/RuBERT классификация отзывов: текст + rating + thumbs_up_count

Тетрадка обучает 3 модели последовательно:

1. `ai-forever/ruBert-large`
2. `ai-forever/sbert_large_mt_nlu_ru` — используется mean pooling
3. `ai-forever/ruRoberta-large`

Пайплайн:

`загрузка данных → train/val/test → tokenizer → Dataset/DataLoader → обучение → best-checkpoint → test metrics → удаление модели из памяти → следующая модель`

Перед запуском положи рядом с ноутбуком обновлённые файлы:

- `dataset.py`
- `model.py`
- `trainer.py`

И укажи ниже правильный путь к папке проекта на Google Drive.

In [1]:
# Если запускаешь в Colab, раскомментируй установку зависимостей при необходимости.
# !pip -q install -U transformers accelerate scikit-learn openpyxl joblib

## 1. Google Drive и пути

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import sys
from pathlib import Path

# Поменяй путь под свою папку на Google Drive.
# В этой папке должны лежать: dataset.py, model.py, trainer.py.
PROJECT_DIR = Path('/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT')
DATA_DIR = PROJECT_DIR / 'data' / 'gold_labeled_data'
CHECKPOINT_DIR = PROJECT_DIR / 'checkpoints'
CHECKPOINT_DIR_RUBERT = PROJECT_DIR / 'checkpoints' / 'rubert_large'
CHECKPOINT_DIR_SBERT = PROJECT_DIR / 'checkpoints' / 'sbert_large'
CHECKPOINT_DIR_ROBERT = PROJECT_DIR / 'checkpoints' / 'ruRoberta_large'
RESULTS_DIR = PROJECT_DIR / 'results'

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
# CHECKPOINT_DIR_RUBERT.mkdir(parents=True, exist_ok=True)
# CHECKPOINT_DIR_SBERT.mkdir(parents=True, exist_ok=True)
# CHECKPOINT_DIR_ROBERT.mkdir(parents=True, exist_ok=True)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_DIR:', DATA_DIR)
print('Files in PROJECT_DIR:', os.listdir(PROJECT_DIR)[:20])

PROJECT_DIR: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT
DATA_DIR: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/data/gold_labeled_data
Files in PROJECT_DIR: ['data', 'dataset.py', 'model.py', 'trainer.py', 'checkpoints', 'results', '__pycache__', 'MY_transformers_3_models_BERT.ipynb']


## 2. Импорты и настройки

In [3]:
import gc
import json
import random
import warnings
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from dataset import ReviewDataset
from model import ModelForClassification
from trainer import Trainer

warnings.filterwarnings('ignore')

RANDOM_SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA memory, GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
CUDA memory, GB: 39.49


## 3. Загрузка текущего датасета

In [4]:
FULL_DATA_XLSX = DATA_DIR / 'app_reviews_ru_gold_labeled.xlsx'
TRAIN_XLSX = DATA_DIR / 'train.xlsx'
TEST_XLSX = DATA_DIR / 'test.xlsx'

DROP_COLS = ['Unnamed: 0', 'Unnamed: 0.1', 'lable_silver_common_AI', 'summary_silver_GPT']


def load_review_data():
    if TRAIN_XLSX.exists() and TEST_XLSX.exists():
        print('Loading existing train.xlsx/test.xlsx')
        train_df = pd.read_excel(TRAIN_XLSX)
        test_df = pd.read_excel(TEST_XLSX)
    else:
        print('train.xlsx/test.xlsx were not found, creating stratified split from FINAL_GOLD')
        df = pd.read_excel(FULL_DATA_XLSX, sheet_name='FINAL_GOLD')
        df = df.drop(columns=DROP_COLS, errors='ignore')
        df = df.dropna(subset=['text', 'label_gold']).reset_index(drop=True)

        train_df, test_df = train_test_split(
            df,
            test_size=0.15,
            random_state=RANDOM_SEED,
            stratify=df['label_gold']
        )

        train_df = train_df.reset_index(drop=True)
        test_df = test_df.reset_index(drop=True)
        train_df.to_excel(TRAIN_XLSX, index=False)
        test_df.to_excel(TEST_XLSX, index=False)

    train_df = train_df.drop(columns=DROP_COLS, errors='ignore')
    test_df = test_df.drop(columns=DROP_COLS, errors='ignore')

    for df in [train_df, test_df]:
        df['text'] = df['text'].fillna('').astype(str)
        df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
        df['thumbs_up_count'] = pd.to_numeric(df['thumbs_up_count'], errors='coerce')

    return train_df, test_df

train_data, test_data = load_review_data()

print('Train:', train_data.shape)
print('Test:', test_data.shape)
print('Train columns:', train_data.columns.tolist())
print('Class distribution:')
print(train_data['label_gold'].value_counts(normalize=True).round(4))
train_data.head()

Loading existing train.xlsx/test.xlsx
Train: (2714, 6)
Test: (480, 6)
Train columns: ['app_name', 'text', 'rating', 'thumbs_up_count', 'label_gold', 'summary_gold']
Class distribution:
label_gold
user_experience    0.4749
bug_report         0.2598
rating             0.2119
feature_request    0.0534
Name: proportion, dtype: float64


,app_name,text,rating,thumbs_up_count,label_gold,summary_gold
0,Avito,программа Авито очень помогает в продаже б.у т...,5,0,user_experience,Положительный опыт использования приложения
1,VK,Скачала и пользовалась с 2015 года. Новый отзы...,1,24,user_experience,Недовольство модерацией и поддержкой
2,Yandex Maps,Сотрудники Яндекс Поддержки продались и сотруд...,1,5,user_experience,Мешает навязчивая реклама
3,Telegram,"прошу, сделайте так чтоб подарки можно было пр...",5,0,feature_request,Разрешить продажу подарков в любое время
4,Yandex Maps,настройте приложение опять заедает и вылетает,1,3,bug_report,Приложение вылетает


## 4. Кодирование меток классов

In [5]:
le = LabelEncoder()
train_data['label_num'] = le.fit_transform(train_data['label_gold'])

if 'label_gold' in test_data.columns:
    test_data['label_num'] = le.transform(test_data['label_gold'])

NUM_CLASSES = len(le.classes_)
label_mapping = {int(i): cls for i, cls in enumerate(le.classes_)}

print('NUM_CLASSES:', NUM_CLASSES)
print('Label mapping:', label_mapping)

with open(RESULTS_DIR / 'label_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(label_mapping, f, ensure_ascii=False, indent=2)

NUM_CLASSES: 4
Label mapping: {0: 'bug_report', 1: 'feature_request', 2: 'rating', 3: 'user_experience'}


## 5. Train/validation split и подготовка `rating/thumbs_up_count`

In [6]:
train_split, val_split = train_test_split(
    train_data,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=train_data['label_num']
)

train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

print('Train split:', train_split.shape)
print('Val split:', val_split.shape)
print('Test:', test_data.shape)

print('Train split class distribution:')
print(train_split['label_gold'].value_counts(normalize=True).round(4))
print('Val split class distribution:')
print(val_split['label_gold'].value_counts(normalize=True).round(4))

Train split: (2171, 7)
Val split: (543, 7)
Test: (480, 7)
Train split class distribution:
label_gold
user_experience    0.4749
bug_report         0.2598
rating             0.2119
feature_request    0.0534
Name: proportion, dtype: float64
Val split class distribution:
label_gold
user_experience    0.4751
bug_report         0.2597
rating             0.2118
feature_request    0.0534
Name: proportion, dtype: float64


In [7]:
def build_meta_raw(df, rating_fill_value=None):
    """
    Возвращает 2 признака:
    1) rating
    2) log1p(thumbs_up_count)

    Масштабирование делается отдельно через StandardScaler,
    который обучается только на train_split.
    """
    meta = df[['rating', 'thumbs_up_count']].copy()

    if rating_fill_value is None:
        rating_fill_value = meta['rating'].median()

    meta['rating'] = meta['rating'].fillna(rating_fill_value)
    meta['thumbs_up_count'] = meta['thumbs_up_count'].fillna(0)
    meta['thumbs_up_count'] = np.log1p(meta['thumbs_up_count'])

    return meta.values.astype('float32')

rating_median = train_split['rating'].median()

train_meta_raw = build_meta_raw(train_split, rating_fill_value=rating_median)
val_meta_raw = build_meta_raw(val_split, rating_fill_value=rating_median)
test_meta_raw = build_meta_raw(test_data, rating_fill_value=rating_median)

meta_scaler = StandardScaler()
train_meta = meta_scaler.fit_transform(train_meta_raw).astype('float32')
val_meta = meta_scaler.transform(val_meta_raw).astype('float32')
test_meta = meta_scaler.transform(test_meta_raw).astype('float32')

joblib.dump(meta_scaler, RESULTS_DIR / 'meta_scaler.joblib')

print('train_meta shape:', train_meta.shape)
print('val_meta shape:', val_meta.shape)
print('test_meta shape:', test_meta.shape)
print('Example meta:', train_meta[:3])

train_meta shape: (2171, 2)
val_meta shape: (543, 2)
test_meta shape: (480, 2)
Example meta: [[-1.0215244   0.47902572]
 [-1.0215244   0.99456733]
 [-1.0215244  -0.40229833]]


## 6. DataLoader factory

In [8]:
def make_dataloaders(tokenizer, max_len, batch_size):
    train_dataset = ReviewDataset(
        train_split,
        tokenizer=tokenizer,
        max_seq_len=max_len,
        text_col='text',
        target_col='label_num',
        meta_features=train_meta,
        use_meta=True,
    )

    val_dataset = ReviewDataset(
        val_split,
        tokenizer=tokenizer,
        max_seq_len=max_len,
        text_col='text',
        target_col='label_num',
        meta_features=val_meta,
        use_meta=True,
    )

    # Если в test есть label_num, метрики будут посчитаны. Если нет — только predictions.
    test_target_col = 'label_num' if 'label_num' in test_data.columns else None
    test_dataset = ReviewDataset(
        test_data,
        tokenizer=tokenizer,
        max_seq_len=max_len,
        text_col='text',
        target_col=test_target_col,
        meta_features=test_meta,
        use_meta=True,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    return train_loader, val_loader, test_loader

## 7. Список моделей

In [9]:
MODEL_SPECS = [
    {
        'model_name': 'ai-forever/ruBert-large',
        'short_name': 'ruBert_large',
        'checkpoint_dir': CHECKPOINT_DIR_RUBERT,
        'pooling': 'cls',
        'max_len': 128,
        'batch_size': 16,
        'lr': 2e-5,
        'n_epochs': 10,
    },
    {
        'model_name': 'ai-forever/sbert_large_mt_nlu_ru',
        'short_name': 'sbert_large_mt_nlu_ru',
        'checkpoint_dir': CHECKPOINT_DIR_SBERT,
        # Для SBERT-подобных моделей лучше использовать mean pooling.
        'pooling': 'mean',
        'max_len': 128,
        'batch_size': 16,
        'lr': 2e-5,
        'n_epochs': 10,
    },
    {
        'model_name': 'ai-forever/ruRoberta-large',
        'short_name': 'ruRoberta_large',
        'checkpoint_dir': CHECKPOINT_DIR_ROBERT,
        'pooling': 'cls',
        'max_len': 128,
        'batch_size': 16,
        'lr': 2e-5,
        'n_epochs': 10,
    },
]

MODEL_SPECS

[{'model_name': 'ai-forever/ruBert-large',
  'short_name': 'ruBert_large',
  'checkpoint_dir': PosixPath('/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/checkpoints/rubert_large'),
  'pooling': 'cls',
  'max_len': 128,
  'batch_size': 16,
  'lr': 2e-05,
  'n_epochs': 10},
 {'model_name': 'ai-forever/sbert_large_mt_nlu_ru',
  'short_name': 'sbert_large_mt_nlu_ru',
  'checkpoint_dir': PosixPath('/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/checkpoints/sbert_large'),
  'pooling': 'mean',
  'max_len': 128,
  'batch_size': 16,
  'lr': 2e-05,
  'n_epochs': 10},
 {'model_name': 'ai-forever/ruRoberta-large',
  'short_name': 'ruRoberta_large',
  'checkpoint_dir': PosixPath('/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/checkpoints/ruRoberta_large'),
  'pooling': 'cls',
  'max_len': 128,
  'batch_size': 16,
  'lr': 2e-05,
  'n_epochs': 10}]

## 8. Функция обучения одной модели

In [11]:
def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def train_and_evaluate_one_model(spec):
    clear_gpu_memory()
    seed_everything(RANDOM_SEED)

    model_name = spec['model_name']
    short_name = spec['short_name']
    pooling = spec['pooling']
    max_len = spec['max_len']
    batch_size = spec['batch_size']
    # checkpoint_dir = spec['checkpoint_dir']

    print('=' * 100)
    print('MODEL:', model_name)
    print('POOLING:', pooling)
    print('MAX_LEN:', max_len, '| BATCH_SIZE:', batch_size)
    print('=' * 100)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_loader, val_loader, test_loader = make_dataloaders(tokenizer, max_len, batch_size)

    # class weights считаем только по train_split
    y_train = train_split['label_num'].values
    weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train),
        y=y_train,
    )
    class_weights = torch.tensor(weights, dtype=torch.float).to(device)
    crit = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.03)

    model_config = {
        'num_classes': NUM_CLASSES,
        'dropout_rate': 0.2,
        'meta_dim': 2,
        'pooling': pooling,
        # Экономит память на больших моделях, но немного замедляет обучение.
        'gradient_checkpointing': True,
    }

    model = ModelForClassification(model_name, config=model_config)

    model_checkpoint_dir = CHECKPOINT_DIR / short_name
    model_checkpoint_dir.mkdir(parents=True, exist_ok=True)

    trainer_config = {
        'lr': spec.get('lr', 2e-5),
        'n_epochs': spec.get('n_epochs', 4),
        'weight_decay': 1e-2,
        'warmup_ratio': 0.1,
        'batch_size': batch_size,
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'seed': RANDOM_SEED,
        'verbose': True,
        'crit': crit,
        'early_stopping': False,
        'patience': 2,
        'save_model_dir': str(model_checkpoint_dir),
        'save_model_name': short_name,
        'save_metric': 'macro_f1',
        'greater_is_better': True,
        'keep_only_best': True,
        'freeze_embeddings': False,
        'freeze_first_n_layers': 0,
        'use_amp': True,
        'max_grad_norm': 1.0,
    }

    trainer = Trainer(trainer_config)
    trainer.fit(model, train_loader, val_loader)

    # Финальная оценка на validation после загрузки лучшего checkpoint.
    val_info = trainer.val_epoch(val_loader)

    # Test predictions.
    test_preds = trainer.predict(test_loader)
    pred_labels = le.inverse_transform(test_preds)

    pred_df = test_data.copy()
    pred_df['pred_label_num'] = test_preds
    pred_df['pred_label_gold'] = pred_labels
    pred_path = RESULTS_DIR / f'predictions_{short_name}.xlsx'
    pred_df.to_excel(pred_path, index=False)

    result = {
        'model': model_name,
        'short_name': short_name,
        'pooling': pooling,
        'best_checkpoint': trainer.best_checkpoint_path,
        'val_loss': val_info['loss'],
        'val_acc': val_info['acc'],
        'val_macro_f1': val_info['macro_f1'],
        'val_weighted_f1': val_info['weighted_f1'],
        'predictions_path': str(pred_path),
    }

    if 'label_num' in test_data.columns:
        y_true = test_data['label_num'].values
        test_acc = accuracy_score(y_true, test_preds)
        test_macro_f1 = f1_score(y_true, test_preds, average='macro')
        test_weighted_f1 = f1_score(y_true, test_preds, average='weighted')

        print('TEST CLASSIFICATION REPORT')
        print(classification_report(y_true, test_preds, target_names=le.classes_))

        result.update({
            'test_acc': test_acc,
            'test_macro_f1': test_macro_f1,
            'test_weighted_f1': test_weighted_f1,
        })

    # Освобождаем память перед следующей моделью.
    del model
    del trainer
    del tokenizer
    del train_loader, val_loader, test_loader
    clear_gpu_memory()

    return result

## 9. Последовательное обучение трёх моделей

In [12]:
all_results = []

for spec in MODEL_SPECS:
    result = train_and_evaluate_one_model(spec)
    all_results.append(result)

    results_df = pd.DataFrame(all_results)
    display(results_df)
    results_df.to_csv(RESULTS_DIR / 'bert_models_results.csv', index=False)

print('Done!')

MODEL: ai-forever/ruBert-large
POOLING: cls
MAX_LEN: 128 | BATCH_SIZE: 16


config.json:   0%|          | 0.00/591 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.78M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

...

Validation: Loss=1.0016; Acc=0.8085; Macro-F1=0.7923; Weighted-F1=0.8081


  0%|          | 0/30 [00:00<?, ?it/s]

TEST CLASSIFICATION REPORT
                 precision    recall  f1-score   support

     bug_report       0.73      0.84      0.78       125
feature_request       0.66      0.73      0.69        26
         rating       0.94      0.87      0.90       101
user_experience       0.86      0.80      0.83       228

       accuracy                           0.82       480
      macro avg       0.80      0.81      0.80       480
   weighted avg       0.83      0.82      0.82       480



,model,short_name,pooling,best_checkpoint,val_loss,val_acc,val_macro_f1,val_weighted_f1,predictions_path,test_acc,test_macro_f1,test_weighted_f1
0,ai-forever/ruBert-large,ruBert_large,cls,/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PR...,0.993802,0.830571,0.810088,0.830382,/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PR...,0.800000,0.777314,0.800746
1,ai-forever/sbert_large_mt_nlu_ru,sbert_large_mt_nlu_ru,mean,/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PR...,0.686881,0.799263,0.791711,0.799793,/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PR...,0.785417,0.775452,0.784918
2,ai-forever/ruRoberta-large,ruRoberta_large,cls,/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PR...,1.001554,0.808471,0.792276,0.808122,/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PR...,0.822917,0.801277,0.824722


Done!


## 10. Сводная таблица результатов

In [13]:
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('test_macro_f1' if 'test_macro_f1' in results_df.columns else 'val_macro_f1', ascending=False)
results_df

,model,short_name,pooling,best_checkpoint,val_loss,val_acc,val_macro_f1,val_weighted_f1,predictions_path,test_acc,test_macro_f1,test_weighted_f1
2,ai-forever/ruRoberta-large,ruRoberta_large,cls,/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PR...,1.001554,0.808471,0.792276,0.808122,/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PR...,0.822917,0.801277,0.824722
0,ai-forever/ruBert-large,ruBert_large,cls,/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PR...,0.993802,0.830571,0.810088,0.830382,/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PR...,0.800000,0.777314,0.800746
1,ai-forever/sbert_large_mt_nlu_ru,sbert_large_mt_nlu_ru,mean,/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PR...,0.686881,0.799263,0.791711,0.799793,/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PR...,0.785417,0.775452,0.784918


## 11. Загрузка лучшей модели при необходимости

In [ ]:
# Пример загрузки лучшего checkpoint одной модели:
# best_path = results_df.iloc[0]['best_checkpoint']
# loaded_trainer = Trainer.load(best_path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
# loaded_model = loaded_trainer.model
# loaded_model.eval()
# print('Loaded:', best_path)

In [14]:
from pathlib import Path
import json
import gc

from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import AutoTokenizer
from torch.utils.data import DataLoader


def get_best_checkpoint_path(short_name, results_table=None):
    """
    Возвращает путь к лучшему checkpoint для модели.

    Приоритет:
    1) колонка best_checkpoint из results_df / bert_models_results.csv;
    2) единственный .ckpt в CHECKPOINT_DIR / short_name;
    3) самый свежий .ckpt в CHECKPOINT_DIR / short_name.
    """
    if results_table is not None and 'best_checkpoint' in results_table.columns:
        rows = results_table[results_table['short_name'] == short_name]
        if len(rows) > 0:
            candidate = rows.iloc[0]['best_checkpoint']
            if pd.notna(candidate) and Path(str(candidate)).exists():
                return Path(str(candidate))

    model_ckpt_dir = CHECKPOINT_DIR / short_name
    ckpt_files = sorted(model_ckpt_dir.glob('*.ckpt'), key=lambda p: p.stat().st_mtime, reverse=True)

    if not ckpt_files:
        raise FileNotFoundError(
            f'Не найден checkpoint для {short_name}. '
            f'Проверил папку: {model_ckpt_dir}'
        )

    return ckpt_files[0]


def make_test_loader_for_model(model_name, max_len, batch_size):
    """
    Создаёт test DataLoader именно для той модели, которую сейчас загружаем.
    Это важно, потому что у разных моделей могут быть разные tokenizer.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    test_target_col = 'label_num' if 'label_num' in test_data.columns else None

    test_dataset = ReviewDataset(
        test_data,
        tokenizer=tokenizer,
        max_seq_len=max_len,
        text_col='text',
        target_col=test_target_col,
        meta_features=test_meta,
        use_meta=True,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    return test_loader, tokenizer


# Если текущая переменная results_df потерялась после перезапуска среды,
# пробуем восстановить её из сохранённого CSV.
results_csv_path = RESULTS_DIR / 'bert_models_results.csv'

if 'results_df' not in globals():
    if results_csv_path.exists():
        results_df = pd.read_csv(results_csv_path)
        print('Загружен results_df из:', results_csv_path)
    else:
        results_df = None
        print('results_df не найден. Checkpoint будут искаться по папкам CHECKPOINT_DIR/<short_name>.')

if 'label_num' not in test_data.columns:
    raise ValueError(
        'В test_data нет колонки label_num, поэтому classification_report построить нельзя. '
        'Проверь test.xlsx или блок кодирования меток.'
    )

comparison_rows = []
reports = {}

for spec in MODEL_SPECS:
    clear_gpu_memory()

    model_name = spec['model_name']
    short_name = spec['short_name']
    max_len = spec.get('max_len', 128)
    batch_size = spec.get('batch_size', 8)

    print('\n' + '=' * 110)
    print(f'Загрузка лучшей модели: {short_name}')
    print(f'HF model: {model_name}')

    best_ckpt_path = get_best_checkpoint_path(short_name, results_df)
    print('Best checkpoint:', best_ckpt_path)

    loaded_trainer = Trainer.load(
        str(best_ckpt_path),
        map_location='cuda' if torch.cuda.is_available() else 'cpu'
    )
    loaded_trainer.verbose = False
    loaded_trainer.model.eval()

    test_loader, tokenizer = make_test_loader_for_model(
        model_name=model_name,
        max_len=max_len,
        batch_size=batch_size,
    )

    y_true = test_data['label_num'].astype(int).values
    y_pred = loaded_trainer.predict(test_loader)

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=list(le.classes_),
        digits=4,
        zero_division=0,
    )

    print('\nCLASSIFICATION REPORT')
    print(report_text)

    test_acc = accuracy_score(y_true, y_pred)
    test_macro_f1 = f1_score(y_true, y_pred, average='macro')
    test_weighted_f1 = f1_score(y_true, y_pred, average='weighted')

    comparison_rows.append({
        'short_name': short_name,
        'model_name': model_name,
        'pooling': spec.get('pooling'),
        'max_len': max_len,
        'batch_size': batch_size,
        'checkpoint': str(best_ckpt_path),
        'test_acc': test_acc,
        'test_macro_f1': test_macro_f1,
        'test_weighted_f1': test_weighted_f1,
    })

    reports[short_name] = report_text

    # Сохраняем predictions по каждой лучшей модели.
    pred_df = test_data.copy()
    pred_df['pred_label_num'] = y_pred
    pred_df['pred_label_gold'] = le.inverse_transform(y_pred)
    pred_df.to_excel(RESULTS_DIR / f'best_checkpoint_test_predictions_{short_name}.xlsx', index=False)

    # Сохраняем текстовый classification_report.
    with open(RESULTS_DIR / f'best_checkpoint_test_report_{short_name}.txt', 'w', encoding='utf-8') as f:
        f.write(f'Model: {model_name}\n')
        f.write(f'Checkpoint: {best_ckpt_path}\n\n')
        f.write(report_text)

    # Освобождаем память перед следующей моделью.
    del loaded_trainer
    del test_loader
    del tokenizer
    gc.collect()
    clear_gpu_memory()

comparison_df = pd.DataFrame(comparison_rows).sort_values('test_macro_f1', ascending=False)
comparison_df.to_csv(RESULTS_DIR / 'best_checkpoints_test_comparison.csv', index=False)

print('\n' + '=' * 110)
print('ИТОГОВОЕ СРАВНЕНИЕ ЛУЧШИХ CHECKPOINT НА TEST')
display(comparison_df)

print('Сохранено:')
print('-', RESULTS_DIR / 'best_checkpoints_test_comparison.csv')
print('-', RESULTS_DIR / 'best_checkpoint_test_report_<model>.txt')
print('-', RESULTS_DIR / 'best_checkpoint_test_predictions_<model>.xlsx')



Загрузка лучшей модели: ruBert_large
HF model: ai-forever/ruBert-large
Best checkpoint: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/checkpoints/ruBert_large/ruBert_large_epoch_08_macro_f1_0_81009.ckpt


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ai-forever/ruBert-large
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



CLASSIFICATION REPORT
                 precision    recall  f1-score   support

     bug_report     0.7429    0.8320    0.7849       125
feature_request     0.6429    0.6923    0.6667        26
         rating     0.8763    0.8416    0.8586       101
user_experience     0.8233    0.7763    0.7991       228

       accuracy                         0.8000       480
      macro avg     0.7713    0.7856    0.7773       480
   weighted avg     0.8037    0.8000    0.8007       480


Загрузка лучшей модели: sbert_large_mt_nlu_ru
HF model: ai-forever/sbert_large_mt_nlu_ru
Best checkpoint: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/checkpoints/sbert_large_mt_nlu_ru/sbert_large_mt_nlu_ru_epoch_02_macro_f1_0_79171.ckpt


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: ai-forever/ruRoberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



CLASSIFICATION REPORT
                 precision    recall  f1-score   support

     bug_report     0.7343    0.8400    0.7836       125
feature_request     0.6552    0.7308    0.6909        26
         rating     0.9362    0.8713    0.9026       101
user_experience     0.8551    0.8026    0.8281       228

       accuracy                         0.8229       480
      macro avg     0.7952    0.8112    0.8013       480
   weighted avg     0.8299    0.8229    0.8247       480


ИТОГОВОЕ СРАВНЕНИЕ ЛУЧШИХ CHECKPOINT НА TEST


,short_name,model_name,pooling,max_len,batch_size,checkpoint,test_acc,test_macro_f1,test_weighted_f1
2,ruRoberta_large,ai-forever/ruRoberta-large,cls,128,16,/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PR...,0.822917,0.801277,0.824722
0,ruBert_large,ai-forever/ruBert-large,cls,128,16,/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PR...,0.800000,0.777314,0.800746
1,sbert_large_mt_nlu_ru,ai-forever/sbert_large_mt_nlu_ru,mean,128,16,/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PR...,0.785417,0.775452,0.784918


Сохранено:
- /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/results/best_checkpoints_test_comparison.csv
- /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/results/best_checkpoint_test_report_<model>.txt
- /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/results/best_checkpoint_test_predictions_<model>.xlsx
